# <span style="color: hotpink;">Spaceship Data Prediction from Kaggle 🛸🛸</span>

In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
result = pd.read_csv('sample_submission.csv')

In [3]:
train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [4]:
train.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
data_info = {}

In [6]:
x = train.drop('Transported', axis = 1)
y = train.Transported.values.astype('int8')

In [7]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)

In [8]:
x_train['Group'] = x_train.PassengerId.apply(lambda x: x.split('_')[0])

In [9]:
x_train['Group_Size'] = x_train.groupby('Group')['Group'].transform('count')

In [10]:
x_train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Group,Group_Size
4278,4558_01,Europa,False,C/167/S,55 Cancri e,54.0,False,0.0,559.0,0.0,15238.0,2799.0,Wezna Baleful,4558,1
5971,6326_01,Earth,False,F/1307/P,TRAPPIST-1e,20.0,False,0.0,20.0,1.0,696.0,0.0,Therek Hinetthews,6326,1
464,0503_02,Mars,False,F/90/S,TRAPPIST-1e,43.0,False,1821.0,0.0,47.0,29.0,0.0,Torms Fone,0503,3
4475,4757_01,Earth,False,F/896/S,TRAPPIST-1e,24.0,False,185.0,0.0,476.0,1810.0,53.0,Tanley Mirandry,4757,1
8469,9046_01,Europa,True,C/335/S,55 Cancri e,25.0,False,0.0,0.0,0.0,0.0,0.0,Alphah Cratrave,9046,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4373,4655_01,Europa,True,B/154/P,55 Cancri e,32.0,False,0.0,0.0,0.0,0.0,0.0,Ainkard Seflock,4655,1
7891,8423_01,Earth,False,F/1620/S,TRAPPIST-1e,22.0,False,0.0,0.0,6.0,0.0,733.0,Analdy Bartez,8423,1
4859,5185_01,Mars,False,E/330/S,TRAPPIST-1e,29.0,False,523.0,0.0,21.0,4.0,811.0,Cruts Flie,5185,1
3264,3499_04,Earth,False,G/574/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Allene Mccarveymon,3499,2


In [11]:
x_train.HomePlanet.value_counts(dropna = False)

HomePlanet
Earth     3651
Europa    1708
Mars      1439
NaN        156
Name: count, dtype: int64

In [12]:
def fillnaproba(x, column_name, one_hot_encoding = False, test = False, data_info = data_info):
    if not test:
        probs = x[column_name].value_counts(normalize=True)
        data_info[f"{column_name}_"] = probs
    else:
        probs = data_info[f"{column_name}_"]  #for prepping x_test using the info from x_train
    
    categories = list(probs.index)
    probabilities = list(probs.values)
    
    missing_mask = x[column_name].isna()
    x.loc[missing_mask, column_name] = np.random.choice(
        categories,
        size=missing_mask.sum(),
        p=probabilities
    )
    
    if one_hot_encoding:
        for cat in categories:
            x[f'{column_name}_{cat}'] = (x[column_name] == cat).astype('int8')
        x.drop(column_name, axis=1, inplace=True)
    
    return x


In [13]:
x_train = fillnaproba(x_train, 'HomePlanet', one_hot_encoding=True)

In [14]:
x_train = fillnaproba(x_train, 'CryoSleep')

In [15]:
x_train.CryoSleep = (x_train.CryoSleep).astype('int8')

In [16]:
x_train.Cabin.value_counts(dropna = False)

Cabin
NaN         151
F/1411/P      7
E/13/S        7
B/11/S        7
F/1808/P      6
           ... 
B/154/P       1
F/1620/S      1
E/330/S       1
G/574/P       1
G/474/P       1
Name: count, Length: 5450, dtype: int64

In [17]:
data_info['Cabin_'] = x_train.Cabin.value_counts() / x_train.Cabin.value_counts().sum()
x_train.Cabin = x_train.Cabin.ffill().bfill()
f = data_info['Cabin_'].keys().map(lambda x: x[0]).unique()
for i in f:
    x_train[f'Deck_{i}'] = (x_train.Cabin.apply(lambda x: x[0]) == i).astype('int8')


In [18]:
x_train['Side'] = x_train.Cabin.apply(lambda x: 1 if x[-1]=='P' else 0)
x_train.drop('Cabin',axis=1,inplace=True)

In [19]:
x_train = fillnaproba(x_train, 'Destination', one_hot_encoding=True)

In [20]:
x_train.columns

Index(['PassengerId', 'CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt',
       'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Group', 'Group_Size',
       'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars', 'Deck_F',
       'Deck_E', 'Deck_B', 'Deck_G', 'Deck_C', 'Deck_A', 'Deck_D', 'Deck_T',
       'Side', 'Destination_TRAPPIST-1e', 'Destination_55 Cancri e',
       'Destination_PSO J318.5-22'],
      dtype='str')

In [21]:
x_train.isna().sum()

PassengerId                    0
CryoSleep                      0
Age                          146
VIP                          176
RoomService                  151
FoodCourt                    148
ShoppingMall                 172
Spa                          152
VRDeck                       146
Name                         156
Group                          0
Group_Size                     0
HomePlanet_Earth               0
HomePlanet_Europa              0
HomePlanet_Mars                0
Deck_F                         0
Deck_E                         0
Deck_B                         0
Deck_G                         0
Deck_C                         0
Deck_A                         0
Deck_D                         0
Deck_T                         0
Side                           0
Destination_TRAPPIST-1e        0
Destination_55 Cancri e        0
Destination_PSO J318.5-22      0
dtype: int64

In [22]:
x_train.Age.describe()

count    6808.000000
mean       28.872944
std        14.481302
min         0.000000
25%        19.000000
50%        27.000000
75%        38.000000
max        79.000000
Name: Age, dtype: float64

In [23]:
x_train.Age = x_train.Age.fillna(28.0)

In [24]:
x_train = fillnaproba(x_train, 'VIP')

In [25]:
x_train.VIP = x_train.VIP.astype('int8')

In [26]:
data_info['RoomServiceMean'] = x_train.RoomService.mean()
data_info['FoodCourtMean'] = x_train.FoodCourt.mean()
data_info['ShoppingMallMean'] = x_train.ShoppingMall.mean()
data_info['SpaMean'] = x_train.Spa.mean()
data_info['VRDeckMean'] = x_train.VRDeck.mean()
x_train.RoomService = x_train.RoomService.fillna(data_info['RoomServiceMean'])
x_train.FoodCourt = x_train.FoodCourt.fillna(data_info['FoodCourtMean'])
x_train.ShoppingMall = x_train.ShoppingMall.fillna(data_info['ShoppingMallMean'])
x_train.Spa = x_train.Spa.fillna(data_info['SpaMean'])
x_train.VRDeck  = x_train.VRDeck.fillna(data_info['VRDeckMean'])

In [27]:
x_train.drop('Name',axis=1,inplace=True)

In [28]:
x_train.drop('PassengerId',axis=1,inplace=True)

In [29]:
x_train.columns

Index(['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall',
       'Spa', 'VRDeck', 'Group', 'Group_Size', 'HomePlanet_Earth',
       'HomePlanet_Europa', 'HomePlanet_Mars', 'Deck_F', 'Deck_E', 'Deck_B',
       'Deck_G', 'Deck_C', 'Deck_A', 'Deck_D', 'Deck_T', 'Side',
       'Destination_TRAPPIST-1e', 'Destination_55 Cancri e',
       'Destination_PSO J318.5-22'],
      dtype='str')

In [30]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)

In [31]:
data_info['sc'] = sc

In [32]:
data_info

{'HomePlanet_': HomePlanet
 Earth     0.53707
 Europa    0.25125
 Mars      0.21168
 Name: proportion, dtype: float64,
 'CryoSleep_': CryoSleep
 False    0.642541
 True     0.357459
 Name: proportion, dtype: float64,
 'Cabin_': Cabin
 F/1411/P    0.001029
 E/13/S      0.001029
 B/11/S      0.001029
 F/1808/P    0.000882
 F/859/P     0.000882
               ...   
 B/154/P     0.000147
 F/1620/S    0.000147
 E/330/S     0.000147
 G/574/P     0.000147
 G/474/P     0.000147
 Name: count, Length: 5449, dtype: float64,
 'Destination_': Destination
 TRAPPIST-1e      0.698163
 55 Cancri e      0.210434
 PSO J318.5-22    0.091403
 Name: proportion, dtype: float64,
 'VIP_': VIP
 False    0.976542
 True     0.023458
 Name: proportion, dtype: float64,
 'RoomServiceMean': np.float64(232.8713802734088),
 'FoodCourtMean': np.float64(454.27725536291507),
 'ShoppingMallMean': np.float64(179.91521675022116),
 'SpaMean': np.float64(308.7857982946192),
 'VRDeckMean': np.float64(302.2806991774383),
 'sc':

In [33]:
def prep(x, data_info):
    x['Group'] = x.PassengerId.apply(lambda x: x.split('_')[0])
    x['Group_Size'] = x.groupby('Group')['Group'].transform('count')
    x = fillnaproba(x, 'HomePlanet', one_hot_encoding=True, test = True)
    x = fillnaproba(x, 'CryoSleep', test = True)
    x.CryoSleep = x.CryoSleep.astype('int8')
    x.Cabin = x.Cabin.ffill().bfill()
    f = data_info['Cabin_'].keys().map(lambda x: x[0]).unique()
    for i in f:
        x[f'Deck_{i}'] = (x.Cabin.apply(lambda x: x[0]) == i).astype('int8')

    
    x['Side'] = x.Cabin.apply(lambda c: 1 if c[-1] == 'P' else 0)
    x.drop('Cabin', axis=1, inplace=True)
    x = fillnaproba(x, 'Destination', one_hot_encoding=True,test = True)
    
    x.Age = x.Age.fillna(28.0)
    
    x = fillnaproba(x, 'VIP', test = True)
    x.VIP = x.VIP.astype('int8')
    x.RoomService = x.RoomService.fillna(data_info['RoomServiceMean'])
    x.FoodCourt = x.FoodCourt.fillna(data_info['FoodCourtMean'])
    x.ShoppingMall = x.ShoppingMall.fillna(data_info['ShoppingMallMean'])
    x.Spa = x.Spa.fillna(data_info['SpaMean'])
    x.VRDeck  = x.VRDeck.fillna(data_info['VRDeckMean'])
    x.drop(['Name', 'PassengerId'], axis=1, inplace=True)
    return x

In [34]:
data_info['Cabin_'].keys()

Index(['F/1411/P', 'E/13/S', 'B/11/S', 'F/1808/P', 'F/859/P', 'G/734/S',
       'B/201/P', 'G/981/S', 'G/1368/P', 'G/974/P',
       ...
       'C/310/S', 'E/190/S', 'D/118/S', 'F/1362/S', 'F/1190/S', 'B/154/P',
       'F/1620/S', 'E/330/S', 'G/574/P', 'G/474/P'],
      dtype='str', name='Cabin', length=5449)

In [35]:
x_test.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name'],
      dtype='str')

In [36]:
x_test = prep(x_test, data_info)

In [37]:
x_test = sc.transform(x_test)

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model = LogisticRegression()
model.fit(x_train, y_train)
accuracy_score(model.predict(x_test), y_test)

0.7814836112708453

In [39]:
def evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    test = accuracy_score(model.predict(x_test), y_test)
    train = accuracy_score(model.predict(x_train), y_train)
    print(f'test accuracy: {test}\n train accuracy: {train}')


In [40]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV


In [55]:
from sklearn.model_selection import GridSearchCV
def dt(x_train,  y_train,x_test = None, y_test = None):

    params_grid = {
        'max_depth': [3, 5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10, 12, 13],
        'min_samples_leaf': [1, 2, 4, 5, 6],
        'criterion': ['gini', 'entropy']
    }
    
    dt = DecisionTreeClassifier(random_state=0)
    gs = GridSearchCV(
        estimator = dt,
        param_grid = params_grid, 
        n_jobs = -1,
        cv = 5
    )
    
    gs.fit(x_train, y_train)
    print(f'Best Parameters: {gs.best_params_}\n Best Score: {gs.best_score_}')
    
    best_modeldt = gs.best_estimator_
    
    
    print(f'train: {accuracy_score(y_train,best_modeldt.predict(x_train))}')
    if (x_test is not None) and (y_test is not None):

        y_pr = best_modeldt.predict(x_test)

        print(f'test: {accuracy_score(y_test, y_pr)}')

In [52]:
def knn(x_train,  y_train,x_test = None, y_test = None):

    knn = KNeighborsClassifier()
    gs2 = GridSearchCV(estimator = knn, 
                      param_grid = {'n_neighbors': list(range(3, 33 , 2))},
                       n_jobs = -1, 
                      cv = 5)
    gs2.fit(x_train, y_train)
    best_knn = gs2.best_estimator_
    best_score = gs2.best_score_
    best_params = gs2.best_params_
    print(f"best score: {best_score}\n best params: {best_params}")
    print(f'train: {accuracy_score(y_train,best_knn.predict(x_train))}')
    if x_test is not None:
        if y_test is not None:
            print(f'test: {accuracy_score(y_test, best_knn.predict(x_test))}')

In [54]:
from xgboost import XGBClassifier
def xg(x_train, y_train, x_test = None, y_test = None):
    params_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0],
    }
    
    xgb = XGBClassifier(random_state=0, eval_metric='logloss')
    
    gs = GridSearchCV(
        estimator=xgb,
        param_grid=params_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    
    gs.fit(x_train, y_train)
    
    print(f'Best Parameters: {gs.best_params_}')
    print(f'Best Score: {gs.best_score_}')
    
    best_modelxg = gs.best_estimator_
    if (x_test is not None) and (y_test is not None):
        print(f'Test Accuracy: {accuracy_score(y_test, best_modelxg.predict(x_test))}')
    print(f'Train Accuracy: {accuracy_score(y_train, best_modelxg.predict(x_train))}')

In [44]:
test

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron
4274,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore
4275,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale


In [45]:
data_info2 = {}
x['Group'] = x.PassengerId.apply(lambda x: x.split('_')[0])
x['Group_Size'] = x.groupby('Group')['Group'].transform('count')
x = fillnaproba(x, 'HomePlanet', one_hot_encoding=True, data_info = data_info2)
x = fillnaproba(x, 'CryoSleep',data_info = data_info2)
x.CryoSleep = (x.CryoSleep).astype('int8')
data_info2['Cabin_'] = x.Cabin.value_counts() / x.Cabin.value_counts().sum()
x.Cabin = x.Cabin.ffill().bfill()
f = data_info2['Cabin_'].keys().map(lambda x: x[0]).unique()
for i in f:
    x[f'Deck_{i}'] = (x.Cabin.apply(lambda x: x[0]) == i).astype('int8')
x['Side'] = x.Cabin.apply(lambda x: 1 if x[-1]=='P' else 0)
x.drop('Cabin',axis=1,inplace=True)
x = fillnaproba(x, 'Destination', one_hot_encoding=True,data_info = data_info2)
x.Age = x.Age.fillna(28.0)
x = fillnaproba(x, 'VIP',data_info = data_info2)
x.VIP = x.VIP.astype('int8')
data_info2['RoomServiceMean'] = x.RoomService.mean()
data_info2['FoodCourtMean'] = x.FoodCourt.mean()
data_info2['ShoppingMallMean'] = x.ShoppingMall.mean()
data_info2['SpaMean'] = x.Spa.mean()
data_info2['VRDeckMean'] = x.VRDeck.mean()
x.RoomService = x.RoomService.fillna(data_info2['RoomServiceMean'])
x.FoodCourt = x.FoodCourt.fillna(data_info2['FoodCourtMean'])
x.ShoppingMall = x.ShoppingMall.fillna(data_info2['ShoppingMallMean'])
x.Spa = x.Spa.fillna(data_info2['SpaMean'])
x.VRDeck  = x.VRDeck.fillna(data_info2['VRDeckMean'])
x.drop('Name',axis=1,inplace=True)
x.drop('PassengerId',axis=1,inplace=True)
sc2 = StandardScaler()
x = sc2.fit_transform(x)
data_info2['sc'] = sc2

In [46]:
test = sc2.transform(prep(test, data_info2))

In [53]:
knn(x_train, y_train, x_test, y_test)

best score: 0.7725065037833142
 best params: {'n_neighbors': 9}
train: 0.8209663503019845
test: 0.7768832662449684


In [58]:
dt(x_train, y_train,x_test, y_test)

Best Parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}
 Best Score: 0.7707792644389162
train: 0.7873166522864539
test: 0.7688326624496837


In [60]:
xg(x_train, y_train, x_test, y_test)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best Score: 0.8038537566783381
Test Accuracy: 0.8021851638872916
Train Accuracy: 0.8346275524877769


In [61]:
knn(x, y)

best score: 0.7700447527810846
 best params: {'n_neighbors': 13}
train: 0.8123777752214425


In [62]:
dt(x, y)

Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 13}
 Best Score: 0.7607318333685152
train: 0.8267571609340849


In [63]:
xg(x, y)

Best Parameters: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best Score: 0.7789070342531156
Train Accuracy: 0.7914413896238353


In [64]:
model = KNeighborsClassifier(n_neighbors = 13)
model.fit(x,y)
prediction = model.predict(test)

In [65]:
final = np.array(list(map(lambda x: True if x==1 else False, prediction)))

In [66]:
result['Transported'] = final

In [67]:
result.to_csv('SpaceshipResult.csv', index = False)

In [68]:
test

array([[ 1.33716309, -0.12629324, -0.15423973, ...,  0.66203165,
        -0.51620026, -0.32272811],
       [-0.74785193, -0.68422342, -0.15423973, ...,  0.66203165,
        -0.51620026, -0.32272811],
       [ 1.33716309,  0.15267185, -0.15423973, ..., -1.51050181,
         1.93723267, -0.32272811],
       ...,
       [ 1.33716309, -0.05655196, -0.15423973, ..., -1.51050181,
         1.93723267, -0.32272811],
       [-0.74785193, -0.05655196, -0.15423973, ..., -1.51050181,
         1.93723267, -0.32272811],
       [ 1.33716309,  0.98956712, -0.15423973, ..., -1.51050181,
        -0.51620026,  3.09858347]], shape=(4277, 25))

In [71]:
print(x.shape)
print(test.shape)

(8693, 25)
(4277, 25)


In [72]:
model2 = XGBClassifier(learning_rate = 0.01, max_depth = 3, n_estimators = 200, subsample = 0.8)
model2.fit(x,y)
prediction2 = model2.predict(test)

In [73]:
final2 = np.array(list(map(lambda x: True if x==1 else False, prediction2)))

In [74]:
result['Transported'] = final2
result.to_csv('SpaceshipResult2.csv', index = False)